# Cold–warm metric — calculations

**What's in this notebook**

- The authoritative cold–warm **trend** calculations (taken verbatim from the original `cw_mean_trend.ipynb`): `dict_percentiles_omip`, `percentiles_omip_trend`, `noaa_sst_trend`, `DA_all`, the perturbation percentile trends, and warm/cold-50% means.
- The cold–warm **gradient / mean / basin** arrays (from `cw_metric.ipynb`) that build on those: `cw_omip`, `cw_noaa`, `cw_cmip`, `cw_mean_omip`, `warm50_omip`, `cold50_omip`, basin warming.
- 20% ranges removed. Calculations match the original ACCESS-OM2 perturbation code exactly.

_Run first; saves everything via `save_state` for the plotting notebooks._


In [22]:
# Load Packages
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import cartopy.crs as ccrs
import calendar, os, re, datetime
from datetime import datetime as _dt
from pathlib import Path
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm, scoreatpercentile
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from xhistogram.xarray import histogram

%matplotlib inline


In [23]:
import sys, os
_p = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_p, 'functions.py')):
        sys.path.insert(0, _p); break
    _p = os.path.dirname(_p)
from functions import *  # shared helper functions


## Authoritative trend calculations (from `cw_mean_trend.ipynb`)


In [24]:
def quantiles_xhist(data):
    hist_T = data.sel(lon = slice(110,290), lat = slice(-20,20))
    weights = np.cos(np.deg2rad(hist_T.lat))
    times = hist_T['time']
    const_area=np.r_[0:1:0.005]
    temps = np.r_[8:34:0.5] # define temperatures for binning 
    
    temps_interp = np.empty([len(times),len(const_area)])
    
    for i in np.r_[0:len(times):1]:  
        h=histogram(hist_T[i],bins=temps,weights=weights).cumsum()
        hh = (h - h[0])/(h[-1]-h[0])
        temps_interp[i]=np.interp(const_area,hh,temps[1:])
    
    
    sst_area= xr.DataArray(temps_interp,coords={'time':times,'area':const_area})

    return(sst_area)



def find_trend_percentile(data):
    data = data.sel(time=slice('1982','2017'))
    x_val = np.arange(0, len(data.time))
    slope_array=[np.polyfit(x_val, data.isel(area=j), 1)[0] for j in range(200)]
    da = xr.DataArray(slope_array, coords ={'area':data.area.values},name = 'trend')
    return da

In [25]:
# omip models - so not access-om2 era5 run

# load percentile data processed in gadi 

mod_labs = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6_iaf'] # ,'ACCESS-OM2-ERA5'

save_directory = '/Users/isabelaconde/Desktop/data/omip/sst_percentiles/'

file_paths = [f"{save_directory}SST_percentile_{model}_1982-2018_gr.nc" for model in mod_labs]

# some datasets have cftime - change in loop below
import datetime
td = datetime.datetime.strptime("1982-01-01", "%Y-%m-%d")
k = 444
d = pd.date_range(td, periods=k, freq='MS')

dict_percentiles_omip = {}

for i, file in enumerate(file_paths):
    dict_percentiles_omip[mod_labs[i]] = xr.open_dataarray(file).assign_coords({'time': d})


# apply function to each model

percentiles_omip_trend = {}
for i, da in enumerate(mod_labs):
    
    percentiles_omip_trend[da]=find_trend_percentile(dict_percentiles_omip[da].sel(time = slice('1982','2017')))
    percentiles_omip_warm50_mean = (dict_percentiles_omip[da].sel(area = slice(0.5,1)).mean(dim = 'area')).mean(dim = 'time')
    percentiles_omip_cold50_mean = (dict_percentiles_omip[da].sel(area = slice(0,0.5)).mean(dim = 'area')).mean(dim = 'time')
    print('trend and mean calculated for ' + mod_labs[i])

# for i in range(len(mod_labs)):
#     # plot trends of highest and lowest model 
#     percentiles_omip_trend[mod_labs[i]].plot(label = mod_labs[i])  





trend and mean calculated for MRI-ESM2-0
trend and mean calculated for FGOALS-f3-L
trend and mean calculated for EC-EARTH3
trend and mean calculated for ACCESS-OM2-JRA55-cycle6_iaf


In [26]:
# cmip modules 
trend_percentile = xr.open_datatree('/Users/isabelaconde/Desktop/data/dt_cmip_percentiles_trend_1982-2018.nc')

# find mods
# Get ensemble names - yes so you can call on it later
models =['CESM1-CAM5',
 'CanESM2',
 'GFDL-ESM2M',
 'MPI-ESM',
 'ACCESS-ESM1-5',
 'CanESM5',
 'EC-Earth3',
 'IPSL-CM6A-LR',
 'MIROC-ES2L',
 'MIROC6',
 'MPI-ESM1-2-LR',
 'CESM2',
 'GFDL-SPEAR-MED']

data_tree=trend_percentile

mods = {}
for model_name, model_data in data_tree.items():
    en=[]
    for ensemble_name, ensemble_data in data_tree[model_name].items():
        en.extend([str(ensemble_name)])
    
    mods[str(model_name)]=en

    
trend_dt=trend_percentile
model_array=[]
da=[]

for m in range(len(models)):
    sources = mods[models[m]][0:len(mods[models[m]])]
    # da=[] # initialise array for grad values
    for i in range(len(sources)):
        da.extend(trend_dt[models[m]].grad.values)

DA_all = xr.DataArray(np.squeeze(da), dims=['run','area'], coords={'run': np.linspace(0, 1, 519), 'area': np.linspace(0, 1, 100)})


# DA_all= xr.open_dataarray('/Users/isabelaconde/Desktop/data/final_data/da_all_cmip56_runs.nc')

def calculate_confidence_intervals_all(data, n):
    mean = np.empty(n)
    lower = np.empty(n)
    upper = np.empty(n)

    for i in range(data.shape[0]):
        mean[i] = np.mean(data[i])
        lower[i] = scoreatpercentile(data[i],10)
        upper[i] = scoreatpercentile(data[i],90)

    return mean, lower, upper

# Plot for first three models
num=100
data = DA_all.values.T * 10 * 12

mean, lower, upper = calculate_confidence_intervals_all(data, num)

df = pd.DataFrame({'mean': DA_all.mean(dim='run').values , 
                   'min': DA_all.min(dim='run').values , 
                   'max': DA_all.max(dim='run').values , 
                   'lower': lower, 'upper': upper,
                   'area': np.linspace(0, 1, 100)})


In [27]:
from xhistogram.xarray import histogram

def percentiles_xhist(data): # find percentiles of SST using xhistogram
    hist_T = data.squeeze()
    weights = np.cos(np.deg2rad(hist_T.lat))
    times = hist_T['time']
    const_area=np.r_[0:1:0.01]
    temps = np.r_[12:34:0.5] # define temperatures for binning 
    
    temps_interp = np.empty([len(times),len(const_area)])
    
    for i in np.r_[0:len(times):1]:  
        h=histogram(hist_T[i],bins=temps,weights=weights).cumsum()
        hh = (h - h[0])/(h[-1]-h[0])
        temps_interp[i]=np.interp(const_area,hh,temps[1:])
    
    
    sst_area= xr.DataArray(temps_interp,coords={'time':times,'area':const_area})

    return(sst_area)

def pacific_mask(sst):

    LonsPts_pos = [130,120,110,110,110,260,270,270,277,291,291,130] #-np.ones(13)*180
    LatsPts_pos =[-20,-10,-5,10,20,20,15,15,10,5,-20,-20]
    
    x_pixel_pos = LonsPts_pos
    y_pixel_pos = LatsPts_pos
    
    temp_list = []
    for a, b in zip(x_pixel_pos, y_pixel_pos):
        temp_list.append([a, b])
    
    polygon = np.array(temp_list)
    
    xv, yv = sst.lon.values, sst.lat.values
    lon2d, lat2d = np.meshgrid(xv, yv)
    points = np.hstack((lon2d.reshape((-1,1)), lat2d.reshape((-1,1))))
    path = matplotlib.path.Path(polygon)
    mask = path.contains_points(points)
    mask.shape = lon2d.shape
    mask_bit_pos = mask*np.ones_like(lon2d)

    mask_mods = mask_bit_pos
    
    mask_mods[mask_mods==0] = np.nan
    return mask_mods


def trend_obs_percentile(data): # find trend of percentile data
    time_axis = np.arange(data.sizes['time'])
    trends = np.empty(data.sizes['area'])
    for i in range(data.sizes['area']):
        y = data[:, i].values
        trends[i] = np.polyfit(time_axis, y, 1)[0]  # slope

    return xr.DataArray(trends, coords={'area': data.area}, dims='area', name='sst_trend')


noaa_sst = xr.open_dataset('/Users/isabelaconde/Desktop/data/conde2025/NOAA_sst.mon.mean.nc').sst.sel(time = slice('1982','2017')).sel(lat = slice(-80,80))
noaa_sst_pacific = (pacific_mask(noaa_sst)*noaa_sst).sel(lat=slice(-20,20), lon = slice(100,300))
noaa_sst_percentiles = percentiles_xhist(noaa_sst_pacific)
noaa_warm50_mean = (noaa_sst_percentiles.sel(area = slice(0.5,1)).mean(dim = 'area')).mean(dim = 'time')
noaa_cold50_mean = (noaa_sst_percentiles.sel(area = slice(0,0.5)).mean(dim = 'area')).mean(dim = 'time')
noaa_sst_trend = trend_obs_percentile(noaa_sst_percentiles)



In [28]:
control_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_control.nc')
era5_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_era5.nc')
perturb_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_merra2_percentile_trend.nc')
# perturb_percent_era5_uas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_percentile_era5_uas_trend.nc')
perturb_percent_era5_uas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/1deg_jra55_iaf_era5_uas_trend_percentile.nc')
perturb_percent_era5_uas_vas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_uas_vas.nc')
perturb_percent_era5_huss_tas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_huss_tas.nc')
perturb_percent_era5_rsds_rlds_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_rsds_rlds.nc')
perturb_percent_era5_winds = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_era5_winds.nc')
perturb_percent_all_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_all_trend.nc')
perturb_percent_winds_mean_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_percentile_trend_uas_vas_mean_trend.nc')


warm50_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_all       = perturb_percent_all_trend.sel(area=slice(0.5, 1)).mean(dim="area")
# warm50_cm2 = cm2_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0.5, 1)).mean(dim="area")
print("perturbation calculated warm 50 mean")

# cold 50 calc
cold50_jra55 = control_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

cold50_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_all       = perturb_percent_all_trend.sel(area=slice(0, 0.5)).mean(dim="area")
# cold50_cm2 = cm2_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold_50_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0, 0.5)).mean(dim="area")
print("perturbation calculated cold 50 mean")

perturbation calculated warm 50 mean
perturbation calculated cold 50 mean


In [29]:
warm50_noaa = noaa_sst_trend.sel(area=slice(0.5, 1)).mean(dim="area")
print("noaa calculated")

warm50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_era5  = era5_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")

cold50_noaa = noaa_sst_trend.sel(area=slice(0, 0.5)).mean(dim="area")
print("noaa calculated")

cold50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")


noaa calculated
noaa calculated


In [30]:

# p = Path("/Users/isabelaconde/Desktop/data/exp_percentiles")
# data = {f.stem: xr.open_dataset(f).to_array().squeeze(drop=True) for f in p.glob("*.nc")}

# warm_means = {k: v.sel(area=slice(0.5, 1)).mean(dim=[d for d in ("time", "area") if d in v.dims]) for k, v in data.items()}
# cold_means = {k: v.sel(area=slice(0, 0.5)).mean(dim=[d for d in ("time", "area") if d in v.dims]) for k, v in data.items()}


In [31]:
warm50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_jra55 = control_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

warm50_era5  = era5_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

## Cold–warm gradient, mean & basin (from `cw_metric.ipynb`)


In [32]:
cw_mean_omip = {}
for i, da in enumerate(mod_labs):
    
    dt = dict_percentiles_omip[da].sel(time = slice('1982','2017'))
    cw_mean_omip[da]=(dt.sel(area = slice(0.5,1)).mean(dim = 'area') - dt.sel(area = slice(0,0.5)).mean(dim = 'area')).mean(dim = 'time')
    print('cw mean for ' + mod_labs[i])




cw mean for MRI-ESM2-0
cw mean for FGOALS-f3-L
cw mean for EC-EARTH3
cw mean for ACCESS-OM2-JRA55-cycle6_iaf


In [33]:
cw_omip = {}
for i, da in enumerate(mod_labs):
    cw_omip[da]=percentiles_omip_trend[da].sel(area = slice(0.5,1)).mean(dim = 'area') - percentiles_omip_trend[da].sel(area = slice(0,0.5)).mean(dim = 'area')
    # print('cw calculated for ' + mod_labs[i])
print('omip cualculated')
# cw_omip
cw_omip 

cw_cmip = DA_all.sel(area = slice(0.5,1)).mean(dim = 'area') - DA_all.sel(area = slice(0,0.5)).mean(dim = 'area')
print('cmip cw calculated')

cw_noaa = noaa_sst_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - noaa_sst_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
print('noaa cw calculated')

cw_jra55 = control_percent_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - control_percent_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_era5 = era5_percent_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - era5_percent_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
# cw_perturb = perturb_percent_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
# cw_perturb_uas = perturb_percent_era5_uas_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_era5_uas_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_perturb_uas_vas = perturb_percent_era5_uas_vas_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_era5_uas_vas_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_perturb_huss_tas = perturb_percent_era5_huss_tas_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_era5_huss_tas_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_era5_rsds_rlds_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_perturb_winds = perturb_percent_era5_winds.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_era5_winds.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_perturb_all = perturb_percent_all_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_all_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
# cw_cm2 = cm2_percent_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - cm2_percent_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
cw_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area = slice(0.5,1)).mean(dim = 'area') - perturb_percent_winds_mean_trend.sel(area = slice(0,0.5)).mean(dim = 'area')
print('perturbation cw calculated')

omip cualculated
cmip cw calculated
noaa cw calculated
perturbation cw calculated


In [34]:
# warmest 50% mean only
warm50_omip = {}
for da in mod_labs:
    warm50_omip[da] = percentiles_omip_trend[da].sel(area=slice(0.5, 1)).mean(dim="area")
print("omip calculated")

warm50_cmip = DA_all.sel(area=slice(0.5, 1)).mean(dim="area")
print("cmip calculated")

warm50_noaa = noaa_sst_trend.sel(area=slice(0.5, 1)).mean(dim="area")
print("noaa calculated")

warm50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_era5  = era5_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")

warm50_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_all       = perturb_percent_all_trend.sel(area=slice(0.5, 1)).mean(dim="area")
# warm50_cm2 = cm2_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
warm50_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0.5, 1)).mean(dim="area")
print("perturbation calculated")


omip calculated
cmip calculated
noaa calculated
perturbation calculated


In [35]:
# coldest 50% mean only
cold50_omip = {}
for da in mod_labs:
    cold50_omip[da] = percentiles_omip_trend[da].sel(area=slice(0, 0.5)).mean(dim="area")
print("omip calculated")

cold50_cmip = DA_all.sel(area=slice(0, 0.5)).mean(dim="area")
print("cmip calculated")

cold50_noaa = noaa_sst_trend.sel(area=slice(0, 0.5)).mean(dim="area")
print("noaa calculated")

cold50_jra55 = control_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

cold50_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0, 0.5)).mean(dim="area")
cold50_perturb_all       = perturb_percent_all_trend.sel(area=slice(0, 0.5)).mean(dim="area")
# cold50_cm2 = cm2_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")
cold_50_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0, 0.5)).mean(dim="area")
print("perturbation calculated")


omip calculated
cmip calculated
noaa calculated
perturbation calculated


In [36]:
cw_mean_cmip = xr.open_dataarray('/Users/isabelaconde/Desktop/data/cw_mean.nc').load()
# cw_mean

In [37]:
folder = "/Users/isabelaconde/Desktop/data/exp_percentiles"

for fname in os.listdir(folder):
    if fname.endswith(".nc"):
        var_name = os.path.splitext(fname)[0]
        globals()[var_name] = xr.open_dataarray(os.path.join(folder, fname))
        print(f"Loaded {var_name}")


Loaded jra55_percentiles
Loaded uas_vas_mean_trend_percentiles
Loaded all_trend_percentiles
Loaded cold_mean
Loaded rsds_rlds_trend_percentiles
Loaded huss_tas_trend_percentiles
Loaded era5_percentiles
Loaded era5_winds_percentiles
Loaded warm_mean
Loaded uas_vas_trend_percentiles


In [38]:
jra55_mean_cw = (jra55_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - jra55_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
era5_mean_cw = (era5_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - era5_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
uas_vas_mean_trend_mean_cw = (uas_vas_mean_trend_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - uas_vas_mean_trend_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
rsds_rlds_trend_mean_cw = (rsds_rlds_trend_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - rsds_rlds_trend_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
huss_tas_trend_mean_cw = (huss_tas_trend_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - huss_tas_trend_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
era5_winds_mean_cw = (era5_winds_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - era5_winds_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
uas_vas_trend_mean_cw = (uas_vas_trend_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - uas_vas_trend_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')


In [39]:
noaa_mean_cw = (noaa_sst_percentiles.sel(area = slice(0.5,1)).mean(dim="area") - noaa_sst_percentiles.sel(area = slice(0,0.5)).mean(dim="area")).mean(dim = 'time')
noaa_mean_cw

<xarray.DataArray ()> Size: 8B
array(3.25491095)

In [40]:
# warmest 50% mean only
basin_omip = {}
for da in mod_labs:
    basin_omip[da] = percentiles_omip_trend[da].sel(area=slice(0, 1)).mean(dim="area")*10*12
print("omip calculated")

basin_cmip = DA_all.sel(area=slice(0, 1)).mean(dim="area")*10*12
print("cmip calculated")

basin_noaa = noaa_sst_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
print("noaa calculated")

basin_jra55 = control_percent_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
basin_era5  = era5_percent_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12

basin_perturb_uas_vas   = perturb_percent_era5_uas_vas_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
basin_perturb_huss_tas  = perturb_percent_era5_huss_tas_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
basin_perturb_rsds_rlds = perturb_percent_era5_rsds_rlds_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
basin_perturb_winds     = perturb_percent_era5_winds.sel(area=slice(0, 1)).mean(dim="area")*10*12
basin_perturb_all       = perturb_percent_all_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
# warm50_cm2 = cm2_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
basin_perturb_winds_mean_trend = perturb_percent_winds_mean_trend.sel(area=slice(0, 1)).mean(dim="area")*10*12
print("perturbation calculated")


omip calculated
cmip calculated
noaa calculated
perturbation calculated


In [41]:
# ACCESS-CM2 percentile trend (for the cold-warm gradient comparison) and CMIP trim arrays
cm2_sst = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS_CM2/cm2__tos_pa.nc')
cm2_percentiles = percentiles_xhist(cm2_sst.rename('sst'))
cm2_percent_trend = trend_obs_percentile(cm2_percentiles)

cold50_cmip_trim = cold50_cmip * 10 * 12
warm50_cmip_trim = warm50_cmip * 10 * 12


In [43]:
# omip_models
# cw_omip
lo = cw_cmip.quantile(0.1, dim="run")
hi = cw_cmip.quantile(0.9, dim="run")

cw_cmip_trim = cw_cmip #.where((cw_cmip >= lo) & (cw_cmip <= hi), drop=True)

df_cw_cmip = (cw_cmip_trim*10*12).to_dataframe(name="value").reset_index()
df_cw_cmip["group"] = "CMIP"


vals = [float(v.values*10*12) for v in cw_omip.values()]   
df_omip = pd.DataFrame({"value": vals})
df_omip["group"] = "OMIP"

df_all = pd.concat([df_cw_cmip[["value","group"]], df_omip], ignore_index=True)
# df_all


# df_mean_cmip = (cw_cmip*10*12).mean(dim='run').to_dataframe('values').reset_index()
# df_mean_omip = np.mean(vals*10*12).to_dataframe('values').reset_index()
# df_all = pd.concat([df_mean_cmip[["value","group"]], df_mean_omip], ignore_index=True)

# means (already scaled)
mean_cmip = (cw_cmip * 10 * 12).mean("run").item()
mean_omip = np.mean(vals)  # vals already scaled in your list comprehension

df_mean_cmip = pd.DataFrame({"value": [mean_cmip], "group": ["CMIP"]})
df_mean_omip = pd.DataFrame({"value": [mean_omip], "group": ["OMIP"]})

df_means = pd.concat([df_mean_cmip, df_mean_omip], ignore_index=True)

In [44]:
# Save all computed variables for the plotting notebooks
save_state(globals())


[save_state] saved 139 variables -> /Users/isabelaconde/Documents/GitHub/JRA55-do_ctrl_ERA5-forcings/processed_data/state.pkl


'/Users/isabelaconde/Documents/GitHub/JRA55-do_ctrl_ERA5-forcings/processed_data/state.pkl'